In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, Paths, and Global Constants
# ═══════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import pandas as pd, numpy as np, json, time, random, itertools
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import warnings; warnings.filterwarnings("ignore")

DATA_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
TRAIN_CSV  = DATA_ROOT / "train.csv"
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME        = "resnet50"
IMAGE_SIZE        = 224
NUM_CLASSES       = 14
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED              = 42
HP_SEARCH_SUBSET = 0.05   # 10% of data for HP search
HP_SEARCH_EPOCHS  = 5      # epochs per trial
FINAL_EPOCHS      = 15     # final training epochs
HP_PATIENCE       = 3      # early stop in HP search
FINAL_PATIENCE    = 5      # early stop in final training
MAX_TRIALS        = 36     # random search cap (Bergstra & Bengio, 2012)

LABEL_NAMES = [
    "No Finding","Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity",
    "Lung Lesion","Edema","Consolidation","Pneumonia","Atelectasis",
    "Pneumothorax","Pleural Effusion","Pleural Other","Fracture","Support Devices"
]
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}")

Device: cuda  |  Model: resnet50


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CheXpert Dataset + Transforms [FIXED]
# ═══════════════════════════════════════════════════════════════════════
from PIL import Image
import numpy as np

def csv_to_abs_path(raw_csv_path, data_root):
    rel = raw_csv_path.replace("CheXpert-v1.0-small/", "")
    return Path(data_root) / rel

def filter_existing_files(df, data_root):
    print("  Building valid patient index from disk (fast scan)...")
    train_dir = Path(data_root) / "train"
    valid_dir = Path(data_root) / "valid"
    existing_patients = set()
    for d in [train_dir, valid_dir]:
        if d.exists():
            for p in d.iterdir():
                if p.is_dir():
                    existing_patients.add(p.name)
    def patient_exists(raw_path):
        parts = raw_path.replace("CheXpert-v1.0-small/", "").split("/")
        return parts[1] if len(parts) > 1 else None
    mask = df["Path"].apply(lambda p: patient_exists(p) in existing_patients)
    n_missing = (~mask).sum()
    print(f"  ✓ {mask.sum():,} valid  |  ✗ {n_missing:,} missing (dropped)")
    return df[mask].reset_index(drop=True)

class CheXpertDataset(Dataset):
    def __init__(self, df, image_root, transform=None, uncertain_policy="ones"):
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(
            -1, 1 if uncertain_policy == "ones" else 0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = csv_to_abs_path(row["Path"], self.image_root)
        img      = Image.open(img_path).convert("RGB")
        lbl      = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        if self.transform: img = self.transform(img)
        return img, lbl

def get_transforms(train=True):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.RandomHorizontalFlip(0.5),
                          T.RandomRotation(10), T.RandomAffine(degrees=0, translate=(0.05,0.05)),
                          T.ColorJitter(brightness=0.1, contrast=0.1), T.ToTensor(),
                          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# ── Load, filter, split ───────────────────────────────────────────────
print("Loading CheXpert data...")
full_df = pd.read_csv(TRAIN_CSV)
print(f"  Raw CSV rows: {len(full_df):,}")

full_df = filter_existing_files(full_df, DATA_ROOT)

hp_df, _         = train_test_split(full_df, test_size=1-HP_SEARCH_SUBSET, random_state=SEED)
hp_train, hp_val = train_test_split(hp_df,   test_size=0.15,               random_state=SEED)
print(f"  HP train: {len(hp_train):,}  |  HP val: {len(hp_val):,}  |  Full (clean): {len(full_df):,}")

Loading CheXpert data...
  Raw CSV rows: 223,414
  Building valid patient index from disk (fast scan)...
  ✓ 223,414 valid  |  ✗ 0 missing (dropped)
  HP train: 9,494  |  HP val: 1,676  |  Full (clean): 223,414


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — Model + Optimizer Builder
# ═══════════════════════════════════════════════════════════════════════
def build_model(hp):
    backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    in_feat  = backbone.fc.in_features
    backbone.fc = nn.Identity()
    drop = hp["dropout"]
    head = nn.Sequential(nn.Dropout(drop), nn.Linear(in_feat, NUM_CLASSES)) if drop > 0 else nn.Linear(in_feat, NUM_CLASSES)
    return nn.Sequential(backbone, head)

def build_opt_sched(model, hp, total_epochs=HP_SEARCH_EPOCHS):
    opt = Adam(model.parameters(), lr=hp["learning_rate"], weight_decay=hp["weight_decay"])
    sch = CosineAnnealingWarmRestarts(opt, T_0=5, T_mult=1, eta_min=1e-6) if hp["use_scheduler"] else None
    return opt, sch

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Class Weight Computation
# ═══════════════════════════════════════════════════════════════════════
def compute_pos_weights(loader):
    pos, total = torch.zeros(NUM_CLASSES), 0
    for _, lbl in loader:
        pos += lbl.sum(0); total += lbl.size(0)
    return (total - pos) / (pos + 1e-5)

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Train / Validate Functions
# ═══════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train(); total_loss = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_epoch(model, loader, criterion):
    model.eval(); total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(imgs); loss = criterion(logits, lbls)
            total_loss += loss.item()
            preds.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(lbls.cpu().numpy())
    p = np.concatenate(preds); l = np.concatenate(labels)
    aurocs = [roc_auc_score(l[:,i],p[:,i]) for i in range(NUM_CLASSES) if l[:,i].sum()>0]
    return total_loss/len(loader), np.mean(aurocs), p, l

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Hardcoded Optimal Config (from completed HP search)
# Trial 16 → AUROC 0.7527 | lr=5e-4, bs=32, wd=1e-5, dropout=0.2, sched=True
# ═══════════════════════════════════════════════════════════════════════

optimal = {
    "learning_rate" : 0.0005,
    "batch_size"    : 32,
    "weight_decay"  : 1e-5,
    "dropout"       : 0.2,
    "use_scheduler" : True,
}
val_auroc_phase1 = 0.7527

print("Optimal HP loaded:")
for k, v in optimal.items():
    print(f"  {k:<20} {v}")
print(f"  Phase 1 best AUROC   {val_auroc_phase1}")

Optimal HP loaded:
  learning_rate        0.0005
  batch_size           32
  weight_decay         1e-05
  dropout              0.2
  use_scheduler        True
  Phase 1 best AUROC   0.7527


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Final Training with Optimal Config
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*68}\n  PHASE 2 — FINAL TRAINING  |  {MODEL_NAME.upper()}  |  {FINAL_EPOCHS} epochs\n{'='*68}")

opt_hp = {k: v for k, v in optimal.items() if k != "val_auroc_phase1"}

full_train, full_val = train_test_split(full_df, test_size=0.05, random_state=SEED)
bs_f  = int(opt_hp["batch_size"])
ft_ds = CheXpertDataset(full_train, DATA_ROOT, get_transforms(True))
fv_ds = CheXpertDataset(full_val,   DATA_ROOT, get_transforms(False))
ft_ld = DataLoader(ft_ds, bs_f, shuffle=True,  num_workers=4, pin_memory=True)
fv_ld = DataLoader(fv_ds, bs_f, shuffle=False, num_workers=4, pin_memory=True)

final_model  = build_model(opt_hp).to(DEVICE)
final_pos_w  = compute_pos_weights(ft_ld).to(DEVICE)
final_crit   = nn.BCEWithLogitsLoss(pos_weight=final_pos_w)
final_opt, final_sch = build_opt_sched(final_model, opt_hp, FINAL_EPOCHS)
final_scaler = torch.amp.GradScaler("cuda")
ckpt_path    = OUTPUT_DIR / f"best_{MODEL_NAME}.pth"

final_log, best_auroc, best_epoch, patience = [], 0.0, 0, 0
for ep in range(1, FINAL_EPOCHS + 1):
    tr_loss              = train_one_epoch(final_model, ft_ld, final_crit, final_opt, final_scaler)
    vl_loss, auroc, _,_  = validate_epoch(final_model, fv_ld, final_crit)
    cur_lr = final_opt.param_groups[0]["lr"]
    if final_sch: final_sch.step()
    final_log.append({"epoch": ep, "train_loss": round(tr_loss,5), "val_loss": round(vl_loss,5),
                      "val_auroc": round(auroc,5), "lr": round(cur_lr,8)})
    marker = ""
    if auroc > best_auroc:
        best_auroc, best_epoch, patience = auroc, ep, 0
        torch.save({"epoch": ep, "model_state_dict": final_model.state_dict(),
                    "auroc": auroc, "optimal_hp": opt_hp, "history": final_log}, ckpt_path)
        marker = " ✓ NEW BEST"
    else:
        patience += 1; marker = f" ({patience}/{FINAL_PATIENCE})"
        if patience >= FINAL_PATIENCE:
            print(f"  ⚠ Early stop at epoch {ep}"); break
    print(f"  Ep{ep:>2}/{FINAL_EPOCHS} | TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} AUROC:{auroc:.4f} LR:{cur_lr:.2e}{marker}")

pd.DataFrame(final_log).to_csv(OUTPUT_DIR / f"final_log_{MODEL_NAME}.csv", index=False)
print(f"\n  Phase 2 done | Best AUROC: {best_auroc:.4f} at epoch {best_epoch}")


  PHASE 2 — FINAL TRAINING  |  RESNET50  |  15 epochs
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 133MB/s]


  Ep 1/15 | TrLoss:0.9621 VlLoss:0.9352 AUROC:0.7471 LR:5.00e-04 ✓ NEW BEST
  Ep 2/15 | TrLoss:0.9260 VlLoss:0.9167 AUROC:0.7598 LR:4.52e-04 ✓ NEW BEST
  Ep 3/15 | TrLoss:0.9046 VlLoss:0.8975 AUROC:0.7722 LR:3.28e-04 ✓ NEW BEST
  Ep 4/15 | TrLoss:0.8815 VlLoss:0.8760 AUROC:0.7818 LR:1.73e-04 ✓ NEW BEST
  Ep 5/15 | TrLoss:0.8617 VlLoss:0.8586 AUROC:0.7904 LR:4.87e-05 ✓ NEW BEST
  Ep 6/15 | TrLoss:0.9019 VlLoss:0.9264 AUROC:0.7678 LR:5.00e-04 (1/5)
  Ep 7/15 | TrLoss:0.8952 VlLoss:0.8919 AUROC:0.7752 LR:4.52e-04 (2/5)
  Ep 8/15 | TrLoss:0.8807 VlLoss:0.8745 AUROC:0.7835 LR:3.28e-04 (3/5)
  Ep 9/15 | TrLoss:0.8629 VlLoss:0.8610 AUROC:0.7908 LR:1.73e-04 ✓ NEW BEST
  Ep10/15 | TrLoss:0.8459 VlLoss:0.8488 AUROC:0.7963 LR:4.87e-05 ✓ NEW BEST
  Ep11/15 | TrLoss:0.8846 VlLoss:0.8840 AUROC:0.7794 LR:5.00e-04 (1/5)
  Ep12/15 | TrLoss:0.8811 VlLoss:0.8883 AUROC:0.7805 LR:4.52e-04 (2/5)
  Ep13/15 | TrLoss:0.8685 VlLoss:0.8710 AUROC:0.7877 LR:3.28e-04 (3/5)
  Ep14/15 | TrLoss:0.8513 VlLoss:0.8556 AU

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Detailed Per-Label Evaluation
# ═══════════════════════════════════════════════════════════════════════
ckpt = torch.load(ckpt_path, weights_only=False)
final_model.load_state_dict(ckpt["model_state_dict"])
_, _, fp, fl = validate_epoch(final_model, fv_ld, final_crit)
print(f"\n{'='*68}\n  FINAL PER-LABEL RESULTS — {MODEL_NAME.upper()}\n{'='*68}")
print(f"  {'Pathology':<30}  {'AUROC':>7}  {'AP':>7}  {'F1':>7}")
per_label = []
for i, name in enumerate(LABEL_NAMES):
    if fl[:,i].sum()==0: continue
    auroc = roc_auc_score(fl[:,i], fp[:,i])
    ap    = average_precision_score(fl[:,i], fp[:,i])
    f1    = f1_score(fl[:,i], (fp[:,i]>=0.5).astype(int), zero_division=0)
    per_label.append({"label":name,"auroc":round(auroc,4),"ap":round(ap,4),"f1":round(f1,4)})
    print(f"  {name:<30}  {auroc:>7.4f}  {ap:>7.4f}  {f1:>7.4f}")
ma = np.mean([r["auroc"] for r in per_label])
print(f"\n  {'MEAN AUROC':<30}  {ma:>7.4f}")
pd.DataFrame(per_label).to_csv(OUTPUT_DIR/f"per_label_{MODEL_NAME}.csv", index=False)
print(f"\n✓ All outputs saved to /kaggle/working/")


  FINAL PER-LABEL RESULTS — RESNET50
  Pathology                         AUROC       AP       F1
  No Finding                       0.8919   0.5053   0.4506
  Enlarged Cardiomediastinum       0.6701   0.2028   0.2495
  Cardiomegaly                     0.8422   0.5557   0.5258
  Lung Opacity                     0.7364   0.7058   0.6788
  Lung Lesion                      0.7967   0.2489   0.1969
  Edema                            0.8530   0.6954   0.6587
  Consolidation                    0.7028   0.3284   0.4045
  Pneumonia                        0.7580   0.3053   0.3324
  Atelectasis                      0.7142   0.4912   0.5422
  Pneumothorax                     0.8495   0.4956   0.4405
  Pleural Effusion                 0.8756   0.8376   0.7735
  Pleural Other                    0.8203   0.1513   0.1482
  Fracture                         0.7985   0.2147   0.1936
  Support Devices                  0.8758   0.8704   0.8214

  MEAN AUROC                       0.7989

✓ All outputs save